# Product Price Prediction: Evaluation, Baselines and Traditional ML

Predicting a product's price from its text description. This notebook sets up an evaluation approach and compares baseline models, from trivial predictors to traditional machine learning.

## Imports

- `LinearRegression`, `RandomForestRegressor`: scikit-learn models.
- `mean_squared_error`, `r2_score`: regression metrics.
- `CountVectorizer`: converts text into word-count vectors.
- `evaluate`, `Item`: helpers from the local `pricer` package. `Item` loads the dataset and `evaluate` scores a pricing function against the test set.

In [ ]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

## Load the data

`LITE_MODE` switches between the smaller and the full dataset. The dataset is pulled from the Hugging Face Hub and returned as train, validation and test splits of `Item` objects.

In [ ]:
LITE_MODE = False

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

## Baseline 1: random pricer

Returns a random price between 1 and 1000, ignoring the item. It sets a floor: any useful model must beat it. The seed is fixed so the result is repeatable.

In [ ]:
def random_pricer(item):
    return random.randrange(1,1000)

In [ ]:
random.seed(42)
evaluate(random_pricer, test)

## Baseline 2: constant pricer

Always predicts the average price of the training set. It still ignores the item, but it is a stronger reference point than random guessing.

In [ ]:
training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

In [ ]:
evaluate(constant_pricer, test)

## Linear regression on hand-crafted features

Three simple features are extracted per item: weight, a flag for missing weight (stored as 0), and the length of the text summary. `list_to_dataframe` turns a list of items into a DataFrame of features plus the target `price`.

In [ ]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

Fits a `LinearRegression` on the training features, prints the learned coefficients and intercept, and reports mean squared error and R-squared on the test set.

In [ ]:
np.random.seed(42)

# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

Wraps the model as a pricing function so it can be scored with the same `evaluate` helper as the baselines.

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [ ]:
evaluate(linear_regression_pricer, test)

## Bag-of-words features

Switches from hand-crafted features to the text itself. `CountVectorizer` builds a vocabulary of the 2,000 most frequent words (excluding English stop words) and represents each description as a vector of word counts.

In [ ]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)

Inspects the vocabulary. `get_feature_names_out()` returns the selected words in alphabetical order, so the slice below shows a sample of 20 of the 2,000.

In [ ]:
selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

## Linear regression on word counts

Fits a linear regression with one coefficient per vocabulary word. The pricing function clamps predictions at 0, since a linear model can output negative prices.

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

In [ ]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [ ]:
evaluate(natural_language_linear_regression_pricer, test)

## Random forest

A random forest is an ensemble of decision trees. A decision tree predicts by applying a sequence of threshold tests on the features, like a flow chart of IF statements. Here the features are word counts, so a tree might test "does 'TV' appear more than 3 times, and 'LED' more than 2?" before assigning a price. Single trees are fast but tend to overfit.

Each tree in the forest is trained on a different random sample of the data and a random subset of features, and the forest averages the predictions of all its trees. `n_estimators=100` sets the number of trees.

Training is limited to the first 15,000 items to keep the run time manageable.

In [ ]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

In [ ]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
evaluate(random_forest, test)

Optional: persist the trained model with `joblib`, which is useful for larger training runs.

In [ ]:
# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## XGBoost

XGBoost is also an ensemble of decision trees, but the trees are built sequentially: each new tree is fitted to correct the errors of the trees before it, using gradient descent. It trains much faster than a random forest, so it is run here on the full training set, and it typically generalizes better.

The import is optional. If it fails, skip this section. On macOS, installing OpenMP may be required: `brew install libomp`.

In [ ]:
import xgboost as xgb

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
evaluate(xg_boost, test)